# Metrics Calculation

This notebook calculates performance metrics (AUC, Precision, Recall, F1, Accuracy) and the Expected Calibration Error (ECE) for multiple cross-validated models: XGBoost, BioBERT, ALBERT, and Med-LLaMA.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm
import os
import json
import shutup; shutup.please()
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
dataset = 'ailf'

In [ ]:
def classification_scores(df, score_column, n_bins=10):
    """
    Calculates the Expected Calibration Error (ECE) of a binary classifier.
    
    Parameters:
        y_true (np.array): True binary labels (0 or 1).
        y_prob (np.array): Predicted probabilities for the positive class.
        n_bins (int): Number of bins for calibration.
        
    Returns:
        float: Expected Calibration Error (ECE) of the classifier.
    """
    y_true = df["label"].values
    y_prob = df[score_column].values
    
    # Ensure inputs are numeric
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Calculate bin edges and indices
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_prob, bin_edges, right=True) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)  # Clip to ensure valid indices
    
    bin_correct_sums = np.zeros(n_bins)
    bin_total_sums = np.zeros(n_bins)
    
    for i in range(n_bins):
        bin_samples = y_true[bin_indices == i]
        bin_total_sums[i] = len(bin_samples)
        bin_correct_sums[i] = bin_samples.sum()
    
    bin_accs = np.divide(bin_correct_sums, bin_total_sums, out=np.zeros_like(bin_correct_sums), where=bin_total_sums != 0)
    bin_confs = np.bincount(bin_indices, weights=y_prob, minlength=n_bins) / (np.bincount(bin_indices, minlength=n_bins) + 1e-12)
    
    ece = np.sum(np.abs(bin_confs - bin_accs) * bin_total_sums) / len(y_true)
    
    auc = roc_auc_score(y_true, y_prob)
    precision = precision_score(y_true, y_prob.round())
    recall = recall_score(y_true, y_prob.round())
    f1 = f1_score(y_true, y_prob.round())
    accuracy = accuracy_score(y_true, y_prob.round())
    
    return [auc, precision, recall, f1, accuracy, ece]

def process_idx(crossval_idx, dataset):
    xgb = pd.read_csv(os.path.join(dataset,"cross_val_xgb", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    xgb.drop("label", axis=1, inplace=True)
    biobert_temp = pd.read_csv(os.path.join(dataset,"cross_val_biobert_temp", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    biobert_temp.drop("label", axis=1, inplace=True)
    albert_temp = pd.read_csv(os.path.join(dataset,"cross_val_albert_temp", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    albert_temp.drop("label", axis=1, inplace=True)
    llama_temp = pd.read_csv(os.path.join(dataset, 'cross_val_llama_temp', f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)

    df = pd.concat([xgb, biobert_temp, albert_temp, llama_temp], axis=1)

    models = df.columns.tolist()
    models.remove("label")

    results = {}
    for model in models:
        results[model] = classification_scores(df, model)

    return results